In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 8.10 Plane Waves and Pseudopotentials

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VIII — Electronic Structure and Many-Body Matter",
    number="8.10",
    title="Plane Waves and Pseudopotentials",
    blurb="The physicist's basis for crystals, priced honestly: Bloch states expanded "
    "in plane waves, the secular matrix assembled from the potential's Fourier "
    "coefficients, gaps opening by exactly twice the coefficient at every folded "
    "crossing, and the empty-lattice skeleton of every fcc band structure drawn "
    "from pure arithmetic. Then the price — a convergence rate proportional to "
    "the potential's softness, measured as a law — and its remedy: a "
    "pseudopotential constructed on the atom, tested for transferability in the "
    "crystal, and honestly broken on a nodal orbital to reveal why real "
    "pseudopotentials must be nonlocal.",
    difficulty="advanced",
    estimate="130–160 min",
)

## Notebook overview

Tight binding ([§8.9](tight-binding.ipynb)) built bands from the atomic limit;
this notebook builds them from the opposite shore. Plane waves are the natural
language of Bloch's theorem, the basis in which the kinetic energy is diagonal
and the potential enters through its Fourier coefficients alone — the basis in
which most of the world's density-functional calculations actually run. Its
virtues are structural: systematic completeness (one knob, the cutoff),
unbiased coverage of space, and machinery that is nothing but the
[§0.6](../00-foundations/fft.ipynb) Fourier toolkit wearing crystal clothes.
Its vice is equally structural: near a nucleus the wavefunction wiggles on the
scale of the core, and resolving wiggles costs plane waves without mercy.

The notebook prices everything. The secular problem is assembled for a
one-dimensional crystal whose Fourier coefficients are *known in closed form*
(the soft-Coulomb chain, whose transform is the Bessel function $2K_0$ met in
[§8.7](kohn-sham.ipynb)); the weak-potential gap law
$E_{\mathrm{gap}} = 2|V_G|$ is verified to a percent and watched failing as the
potential strengthens. The empty-lattice bands of the fcc crystal are folded
along $L$–$\Gamma$–$X$ by pure arithmetic — the skeleton under every real fcc
band structure, degeneracies counted shell by shell. The cost of hardness
becomes a measured law: exponential convergence with a rate *proportional to
the potential's softening length* (fitted slopes $-2.95, -1.92, -1.38$ for
$\varepsilon = 0.5, 0.3, 0.2$). And the remedy is built the way practitioners
build it: a pseudopotential *constructed on the atom* (match the valence
level and its tail, banish the core state), then *tested in the crystal* —
where it transfers beautifully for an $s$-like valence (band shapes
correlating at $+0.999$ across lattice constants) and fails with instructive
perfection for a nodal one (correlation $-0.996$: a local potential cannot
know angular character), motivating the nonlocal projectors every modern
pseudopotential carries and setting up the empirical shortcut of
[§8.11](epm-band-structures.ipynb).

> **Conventions (this notebook).** Hartree atomic units. The one-dimensional
> crystal is the soft-Coulomb chain $V(x) = -Z\sum_n [ (x-na)^2 +
> \varepsilon^2 ]^{-1/2}$ with Fourier coefficients $V_G = -(2Z/a)\,
> K_0(|G|\varepsilon)$ for $G \ne 0$ (`scipy.special.k0`); the divergent
> $G = 0$ coefficient (the one-dimensional Coulomb tail) is dropped, so all
> crystal energies carry an overall alignment constant and band *shapes* are
> the physical content. Secular matrices are explicit `numpy` arrays over
> $G \in (2\pi/a)[-n_G, n_G]$, diagonalized by `numpy.linalg.eigvalsh`.
> Atomic reference problems use the [§8.2](exact-laboratory.ipynb)-style grid
> and `scipy.linalg.eigh_tridiagonal`.
>
> **How to read the checks.** Each exercise closes with a `validate` call
> against an independent fact: a perturbative law, exact degeneracy counts, a
> fitted rate, a correlation coefficient. A ✓ is strong evidence; a ✗ is a
> prompt to locate the discrepancy, not an automatic verdict.
>
> **Scope.** Local pseudopotentials only, constructed by level-and-tail
> matching; norm conservation, Kleinman–Bylander separable forms, ultrasoft
> and PAW machinery are surveyed in Martin {cite}`martin2004`, Ch. 11, and
> Giustino {cite}`giustino2014`, Ch. 5 — and the reason they exist is
> precisely the nodal lesson this notebook closes on.

## Theory in brief

### The secular problem in plane waves

Bloch's theorem
([§7.12](../07-quantum-statistical-mechanics/bloch-theorem-band-structure.ipynb))
writes a crystal state as $e^{ikx}$ times a lattice-periodic function, and
expanding the periodic part in reciprocal-lattice harmonics turns the
Schrödinger equation into linear algebra:

```{math}
:label: eq-pw-secular
\sum_{G'} \left[ \tfrac12 (k+G)^2\,\delta_{GG'} + V_{G-G'} \right] c_{G'}
= \varepsilon(k)\, c_G ,
```

one matrix per $k$, kinetic energy on the diagonal, the potential a Toeplitz
pattern of Fourier coefficients. Two limits organize everything. With
$V = 0$ the eigenvalues are free parabolas *folded* into the first zone —
every band structure's skeleton. With $V$ weak, degenerate folded branches
split by first-order degenerate perturbation theory
([§6.21](../06-quantum-mechanics/perturbation-fine-structure.ipynb)): at the
zone boundary the two branches $k$ and $k - 2\pi/a$ mix through exactly one
coefficient, and

```{math}
:label: eq-pw-gap
E_{\mathrm{gap}} = 2\,|V_{G}|
\qquad (G = 2\pi/a,\ \text{weak } V):
```

the gap *is* the Fourier coefficient, doubled. Everything a crystal does to
electrons at band edges is encoded in how its potential decomposes into
harmonics.

### The cost of hardness

How many plane waves? The coefficients of the soft-Coulomb chain decay as
$K_0(|G|\varepsilon) \sim e^{-|G|\varepsilon}$: the softening length
$\varepsilon$ sets the reciprocal-space extent of the potential, and with it
the basis size. The rule — convergence exponential with rate proportional to
the softness — is measured below as a fitted law, and its message is the
plane-wave method's economics in one line: *core wiggles are unaffordable;
soften them*. That is the entire mandate of the pseudopotential.

### Pseudopotentials: construction, transferability, and the nodal catch

Valence electrons decide bonding; core states merely enforce orthogonality,
and their principal effect on valence orbitals is the nodes they impose near
the nucleus. The pseudopotential program (Martin {cite}`martin2004`, Ch. 11)
replaces nucleus-plus-core by a soft effective potential built on the *atom*:
choose a smooth candidate, tune its parameters so its ground level matches
the true valence level and its wavefunction matches the true one outside a
core radius $r_c$, and — critically — *test* it in environments it was not
fitted to. That last property, transferability, is what makes
pseudopotentials predictive rather than descriptive, and it works because
scattering at the valence energy is controlled by the wavefunction outside
$r_c$, which the construction preserves. The catch, delivered honestly in the
closing exercise: a *local* potential's crystal band inherits the
nodal/angular character of the orbital it carries, and no tuning of a local
well can turn an $s$-like band into a $p$-like one. Real pseudopotentials are
therefore **nonlocal** — a different potential per angular channel, applied
through projectors — and the one-dimensional laboratory makes the necessity
measurable as a sign.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh_tridiagonal
from scipy.optimize import least_squares
from scipy.special import k0

from ecp import validate

INK, AMBER, SOFT = "#16213e", "#c0851a", "#46506b"


def pw_bands(a_lat, z_chg, eps_soft, n_g, k_list, n_bands=4):
    """Plane-wave band structure of the 1-D soft-Coulomb chain.

    Assembles the secular matrix of Eq. eq-pw-secular with the closed-form
    coefficients V_G = -(2Z/a) K0(|G| eps) (the Bessel transform of the
    soft-Coulomb well, section 8.7) and diagonalizes per k. The divergent
    G = 0 coefficient is dropped: all energies carry one alignment constant.

    Parameters
    ----------
    a_lat : float
        Lattice constant (Bohr).
    z_chg : float
        Well charge Z.
    eps_soft : float
        Softening length (Bohr); sets the Fourier decay e^(-|G| eps).
    n_g : int
        Reciprocal vectors run over (2 pi/a) [-n_g, n_g].
    k_list : array-like
        Crystal momenta to solve at.
    n_bands : int, optional
        Bands returned per k.

    Returns
    -------
    numpy.ndarray
        Array of shape (len(k_list), n_bands) of band energies (Hartree).
    """
    g_vecs = 2.0 * np.pi / a_lat * np.arange(-n_g, n_g + 1)
    out = []
    for k in k_list:
        ham = np.diag(0.5 * (k + g_vecs) ** 2)
        for i, g_i in enumerate(g_vecs):
            for j, g_j in enumerate(g_vecs):
                dg = abs(g_i - g_j)
                if dg > 1e-12:
                    ham[i, j] += -(2.0 * z_chg / a_lat) * k0(dg * eps_soft)
        out.append(np.linalg.eigvalsh(ham)[:n_bands])
    return np.array(out)


# atomic reference grid for the pseudopotential construction
x_atom = np.linspace(-15.0, 15.0, 3001)
h_atom = x_atom[1] - x_atom[0]
OFF_ATOM = np.full(3000, -0.5 / h_atom**2)


def atom_levels(z_chg, eps_soft, n_states=3):
    """Bound levels and orbitals of the single soft-Coulomb well.

    Parameters
    ----------
    z_chg, eps_soft : float
        Well parameters of -Z/sqrt(x^2 + eps^2).
    n_states : int, optional
        Number of lowest states.

    Returns
    -------
    tuple
        ``(energies, orbitals)``: orbitals normalized on the grid.
    """
    energies, vecs = eigh_tridiagonal(
        1.0 / h_atom**2 - z_chg / np.sqrt(x_atom**2 + eps_soft**2),
        OFF_ATOM,
        select="i",
        select_range=(0, n_states - 1),
    )
    return energies, vecs / np.sqrt(h_atom)

## Exercise 1 — The gap is a Fourier coefficient

Equation {eq}`eq-pw-gap` is degenerate perturbation theory's cleanest
prediction, and the soft-Coulomb chain's closed-form coefficients make it a
two-line check: at the zone boundary $k = \pi/a$ the gap between bands 1 and
2 should equal $2|V_{2\pi/a}| = (4Z/a)\,K_0(2\pi\varepsilon/a)$ while the
potential is weak, and overshoot it as higher orders kick in.

**Part a)** For the chain with $a = 2.5$, $\varepsilon = 0.5$, compute the
zone-boundary gap from the Setup helper `pw_bands` ($n_G = 12$) at
$Z = 0.05$ and $Z = 2.0$, against the first-order prediction (a `scipy.special.k0`
evaluation).

**Part b)** Sweep $Z$ from $0.05$ to $2$ and plot the measured-to-predicted
ratio: unity at weak coupling, drifting beyond it as the perturbative regime
ends — a validity boundary located by computation rather than decree.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1 — first order, then not

The weak-coupling ratio must be 1 within a percent, and the strong-coupling
ratio must have drifted measurably (beyond $2\%$): the law and its limits in
two numbers.

In [ ]:
validate.close(float(ratios_gap[0]), 1.0, "the 2|V_G| law at weak coupling", rtol=1e-2)
validate.check(
    abs(ratios_gap[-1] - 1.0) > 0.02,
    "higher orders visible at strong coupling",
    f"ratio {ratios_gap[-1]:.3f} at Z = 2",
)

## Exercise 2 — Folding, and gaps at every crossing

The empty lattice is the skeleton: at $V = 0$ Eq. {eq}`eq-pw-secular` gives
the free parabolas $\tfrac12(k+G)^2$, one branch per reciprocal vector,
folded into the first zone. Switching the potential on splits every folded
crossing.

**Part a)** Plot the lowest four folded free branches over
$k \in [0, \pi/a]$ (plain `numpy` arithmetic on $\tfrac12(k+G)^2$), and
confirm the band-1/band-2 crossing sits exactly at the zone boundary with the
free value $\tfrac12(\pi/a)^2$.

**Part b)** Overlay the true bands at $Z = 1$ (the Setup helper): gaps open
precisely where branches crossed and nowhere else — avoided crossings, the
[§6.21](../06-quantum-mechanics/perturbation-fine-structure.ipynb) mechanism
drawn at crystal scale.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2 — gaps live on crossings

The sorted free branches must be degenerate at the zone boundary (bands 1–2)
and at $\Gamma$ (bands 2–3), and the true bands must split both while
tracking the skeleton over the inner 60% of the zone (band 1 within a
constant offset of the free branch to $15\%$ of the inner bandwidth; nearer
the boundary the gap physics itself bends the band away, as it must).

In [ ]:
free_sorted = np.sort(free_branches, axis=1)
validate.close(
    float(free_sorted[-1, 1] - free_sorted[-1, 0]),
    0.0,
    "free degeneracy at the zone boundary",
    rtol=0.0,
    atol=1e-9,
)
gap_12 = float(true_bands[-1, 1] - true_bands[-1, 0])
validate.check(gap_12 > 0.05, "the crossing opens into a gap", f"gap = {gap_12:.3f} Ha")
# k_path starts at 1e-4, not 0, so the free 2-3 split there is 2kG ~ 5e-4:
# hence the loose atol on the Gamma degeneracy.
validate.close(
    float(free_sorted[0, 2] - free_sorted[0, 1]),
    0.0,
    "free degeneracy at Gamma",
    rtol=0.0,
    atol=1e-3,
)
gap_23 = float(true_bands[0, 2] - true_bands[0, 1])
validate.check(
    gap_23 > 0.05, "the Gamma crossing opens into a gap", f"gap = {gap_23:.3f} Ha"
)
# the tracking claim holds AWAY from the boundary: near it, the very gap
# physics being celebrated bends the band off the parabola, so the window
# stops at 60% of the zone.
inner = k_path <= 0.6 * np.pi / A_LAT
disp_free = free_sorted[inner, 0] - free_sorted[0, 0]
disp_true = true_bands[inner, 0] - true_bands[0, 0]
validate.close(
    float(np.abs(disp_true - disp_free).max()) / float(np.ptp(disp_free)),
    0.0,
    "inner-zone dispersion tracks the skeleton",
    rtol=0.0,
    atol=0.15,
)

## Exercise 3 — The fcc skeleton: every textbook's first band structure

In three dimensions the same folding produces the celebrated empty-lattice
fcc diagram — the skeleton under silicon, aluminum, and every other fcc
band structure, including the real ones of
[§8.11](epm-band-structures.ipynb). The fcc reciprocal lattice is bcc,
$\mathbf G = \tfrac{2\pi}{a}(n_1, n_2, n_3)$ with all-even or all-odd
integers, and the branches are $\tfrac12|\mathbf k + \mathbf G|^2$ along
$L = \tfrac{\pi}{a}(1,1,1) \to \Gamma \to X = \tfrac{2\pi}{a}(1,0,0)$.

**Part a)** Generate the bcc reciprocal set by integer enumeration
(`numpy` meshgrid over $n_i \in [-2, 2]$ filtered by
the all-even/all-odd rule), and count the degeneracies of the free levels at
$\Gamma$: the shells $(0,0,0)$, $(1,1,1)$-type, and $(2,0,0)$-type must
count $1, 8, 6$.

**Part b)** Plot the folded branches along $L$–$\Gamma$–$X$ (in units of
$(2\pi/a)^2$): the classic figure, and the answer key for reading any real
fcc band structure — every feature of silicon's valence bands in
[§8.11](epm-band-structures.ipynb) is one of these branches, gapped.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3 — the shells count themselves

The $\Gamma$-point degeneracies must be exactly $1, 8, 6$ (integer
arithmetic), the lowest branch at $\Gamma$ exactly zero, and the lowest
level at $X$ exactly $1/2$ in the reduced units (the $(1,0,0)$ zone-boundary
free energy).

In [ ]:
validate.check(
    shell_counts[0] == 1 and shell_counts[3] == 8 and shell_counts[4] == 6,
    "fcc shell degeneracies 1, 8, 6 at Gamma",
    f"{shell_counts[0]}, {shell_counts[3]}, {shell_counts[4]}",
)
validate.close(
    float(branches[100, 0]), 0.0, "the lowest level at Gamma", rtol=0.0, atol=1e-12
)
validate.close(float(branches[-1, 0]), 0.5, "the lowest level at X", rtol=1e-12)

## Exercise 4 — The cost of hardness, as a law

The coefficients $V_G \propto K_0(|G|\varepsilon)$ decay exponentially with
the softening length $\varepsilon$, so the basis error should too — and the
*rate* should scale with $\varepsilon$ itself. A claim about rates is a fit.

**Part a)** For the chain at $a = 2.5$, $Z = 5$, $k = 0.4$: compute the
band-1 error against an $n_G = 40$ reference for $n_G = 3, 5, 7, 9$, at the
three softenings $\varepsilon = 0.5, 0.3, 0.2$, and fit each error sequence's
logarithm against $n_G$ (`numpy.polyfit`).

**Part b)** Plot the three convergence lines and compare the fitted rates:
their ratios must track the $\varepsilon$ ratios ($0.6$ and $0.4$ against the
$\varepsilon = 0.5$ baseline) — *convergence rate proportional to softness*,
the plane-wave method's economics in one measured law, and the entire
mandate of the pseudopotential in one figure.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4 — the economics, fitted

All three convergences must be genuinely exponential (monotone falling by
orders of magnitude), and the rate ratios must track the softness ratios
within $20\%$ — the measured law behind every plane-wave cutoff ever chosen.

In [ ]:
validate.check(
    all(bool(np.all(np.diff(errors_all[e_v]) < 0)) for e_v in (0.5, 0.3, 0.2)),
    "exponential convergence at every hardness",
    "errors fall monotonically over decades",
)
validate.close(ratio_03, 0.6, "rate ratio tracks softness ratio (0.3/0.5)", rtol=0.2)
validate.close(ratio_02, 0.4, "rate ratio tracks softness ratio (0.2/0.5)", rtol=0.2)

## Exercise 5 — A pseudopotential, built on the atom and tested in the crystal

The construction follows the practitioner's liturgy. The all-electron atom is
the single well $-Z/\sqrt{x^2+\varepsilon^2}$ with $Z = 5$,
$\varepsilon = 0.2$: levels $-16.74$ (core), $-6.87$, and $-3.79$ Ha. The
target valence state is the *even* level at $-3.787$ Ha (two nodes; its
even parity matches a nodeless pseudo ground state — the odd level's turn
comes in Exercise 6). The pseudopotential is the same functional form, made
soft, with $(Z_{\mathrm{pp}}, \varepsilon_{\mathrm{pp}})$ tuned so its
nodeless ground state reproduces the valence *energy* and the valence
*amplitude* at the matching radius $r_c = 3$ Bohr. The choice of $r_c$ is
itself physics: crystal bandwidths are set by wavefunction overlap at the
*neighbor distance*, so the tail must be matched out to roughly $a/2$ of the
environments one intends to predict — a matching radius of 2 Bohr reproduces
the level perfectly yet misses the $a = 5$ bandwidth by 70%, while $r_c = 3$
brings both test crystals within 25% (the solution prints the comparison).

**Part a)** Fit with `scipy.optimize.least_squares` (residuals: the level
mismatch, and $10\times$ the amplitude mismatch at $r_c$; bounds
$Z \in [0.1, 15]$, $\varepsilon \in [0.3, 5]$; the Setup helper
`atom_levels`). Confirm the pseudo atom has *no* state near the core level.

**Part b)** The transferability test: place both atoms in crystals at
$a = 5$ and $a = 6$ (environments the fit never saw) and compare the pseudo
band 1 against the all-electron band 3 (the target level's band), after mean
alignment (the dropped $V_0$'s constant). Correlation and width agreement
across both lattice constants — a fit at one geometry predicting two others —
is the property that makes the entire pseudopotential industry possible.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5 — transferability, earned

The pseudo ground level must match the target valence level to $10^{-2}$ Ha;
both crystal tests must correlate above $+0.99$; and the band widths must
agree within $25\%$ at both lattice constants — prediction, not description.

In [ ]:
validate.close(
    float(ps_levels[0]),
    float(target_energy),
    "the atomic valence level, matched",
    rtol=0.0,
    atol=1e-2,
)
for a_test in (5.0, 6.0):
    corr, w_ae, w_ps = transfer_stats[a_test]
    validate.check(
        corr > 0.99, f"band shapes correlate at a = {a_test}", f"corr {corr:+.4f}"
    )
    validate.close(w_ps, w_ae, f"bandwidths agree at a = {a_test}", rtol=0.25)

## Exercise 6 — The nodal catch: why pseudopotentials are nonlocal

Now the same liturgy on the *odd* valence level ($-6.87$ Ha, one node), and
the honest failure that built an industry. A nodeless pseudo ground state can
match the odd level's energy and tail perfectly on the atom — but in the
crystal, the band's *dispersion sign* is set by the orbital's parity
(odd orbitals flip the effective hopping's sign, exactly the
[§8.9](tight-binding.ipynb) overlap logic), and no local potential can give
a nodeless orbital an odd orbital's band.

**Part a)** Refit the pseudopotential against the odd level (the same
residuals with the $-6.87$ Ha target and its amplitude at $r_c = 1.5$), and
confirm the atomic match is again excellent.

**Part b)** Run the crystal test at $a = 3$: the aligned bands must
*anti*-correlate (measured $-0.996$) — the pseudo band curves the wrong way,
with near-perfect magnitude and reversed sign. The remedy, stated plainly:
real pseudopotentials are **nonlocal**, one potential per angular-momentum
channel applied through projectors, so each channel's nodal character is
respected (Martin {cite}`martin2004`, Ch. 11). The empirical method of
[§8.11](epm-band-structures.ipynb) sidesteps the construction entirely by
fitting the crystal's Fourier coefficients to experiment — and inherits this
lesson as its license.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6 — the minus sign that built an industry

The odd-level atomic fit must match the level to $10^{-2}$ Ha, and the
crystal bands must anti-correlate below $-0.9$: a perfect atomic fit and a
reversed crystal band, in the same construction.

In [ ]:
validate.close(
    float(atom_levels(Z_ODD, EPS_ODD, 1)[0][0]),
    float(target_odd),
    "the odd valence level, matched on the atom",
    rtol=0.0,
    atol=1e-2,
)
validate.check(
    corr_odd < -0.9,
    "the crystal band anti-correlates (the nodal sign)",
    f"corr {corr_odd:+.4f}",
)

:::{admonition} With your assistant
:class: tip
The secular builder `pw_bands` recomputes every $K_0$ per matrix element —
clean but wasteful, since $V_{G-G'}$ depends only on the difference. Have
your assistant refactor it to precompute the coefficient vector once and
fill the matrix by `numpy` index arithmetic (a Toeplitz fill), then run the
check that is yours alone: the refactored bands must match the original at
machine precision across ten random $(k, Z, \varepsilon)$ triples
(`numpy.allclose`, `rtol=1e-12`) while running measurably faster
(`time.perf_counter` ratio). The check is yours.
:::

## Notebook summary

The plane-wave method arrived with its price tag attached. The secular
problem's weak-coupling gap law $E_{\mathrm{gap}} = 2|V_G|$ held to $0.1\%$
at $Z = 0.05$ and drifted measurably by $Z = 2$; folding produced the
skeleton (gaps only where free branches crossed), and the fcc empty lattice
delivered the classic $L$–$\Gamma$–$X$ diagram with its shell degeneracies
$1, 8, 6$ counted by integer arithmetic. The cost of hardness became a
fitted law — exponential convergence with rate proportional to the
softening ($-2.95, -1.92, -1.38$ at $\varepsilon = 0.5, 0.3, 0.2$; rate
ratios $0.65, 0.47$ against softness ratios $0.6, 0.4$). The pseudopotential
answered: built on the atom (valence level matched to $10^{-4}$, tail
matched at $r_c$), it transferred to crystals the fit never saw with band
correlations $+0.999$ and widths within 25% — and then failed with
perfect instructiveness on the odd level, whose crystal band anti-correlated
at $-0.996$ because parity sets curvature and no local well can fake a node.
The minus sign is the reason every modern pseudopotential is nonlocal, and
the license for the empirical shortcut that now builds real silicon.

## Outlook

- [§8.11](epm-band-structures.ipynb) takes the shortcut: rather than
  constructing potentials from atoms, the empirical pseudopotential method
  fits the crystal's three symmetric (and, for GaAs, three antisymmetric)
  Fourier coefficients directly to measured gaps — and the full band
  structures of real silicon and gallium arsenide fall out of a matrix this
  notebook already knows how to build.
- Norm conservation (making the pseudo orbital's charge inside $r_c$ match
  the true one) upgrades level-and-tail matching into energy-derivative
  matching, the modern transferability criterion; Kleinman–Bylander
  separability makes nonlocality affordable; PAW keeps the true wiggles
  recoverable. Martin {cite}`martin2004`, Ch. 11, walks the whole ladder.
- The hardness law is why "soft" and "ultrasoft" are words worth millions of
  core-hours: halving the effective hardness halves the exponent that sets
  every plane-wave calculation's memory and time.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()